# Milestone 3 — Training EN→ES Transformer on a T4

Trains two capacity presets (37.6M and 24.0M parameters) and compares greedy
vs beam decoding.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

Checkpoints are written to Google Drive after every epoch, so a killed session
resumes instead of restarting.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("!! No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 1. Mount Google Drive

Checkpoints survive session death only if they live here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/nmt-en-es'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
!mkdir -p "{CKPT_DIR}"
print('checkpoints ->', CKPT_DIR)

## 2. Clone the project

In [ ]:
!git clone https://github.com/halxbrown/nmt-en-es.git /content/nmt-en-es
%cd /content/nmt-en-es
!ls

In [ ]:
!pip install -q sentencepiece sacrebleu rouge-score 'datasets>=3.0'
# Do NOT reinstall torch -- Colab's build is already CUDA-matched.

## 3. Rebuild data (Milestone 1) and verify masks (Milestone 2)

`data/` and the `.npz` caches are gitignored, so the corpus has to be rebuilt
here. Colab's connection downloads it much faster than a home line.

These should reproduce the local numbers exactly: 78.0% padding efficiency,
four mask PASSes, and both behavioural tests at 0.000e+00.

In [ ]:
!python run_milestone1.py --force

In [ ]:
!python run_milestone2.py --steps 60

## 4. Train both presets

`base` = 37.6M params (4+4 layers, d_ff 2048), `small` = 24.0M (3+3, d_ff 1024).
Dropout is held at 0.2 for both so the ablation isolates capacity.

Roughly 1–3 min/epoch on a T4, up to 20 epochs each with early stopping.
Budget 30–90 minutes and keep the tab open.

**If the session dies, just rerun this cell** — it resumes from `last.pt` with
optimizer and scheduler state intact.

In [ ]:
!python run_milestone3.py --preset both --epochs 20 --checkpoint-dir "{CKPT_DIR}"

## 5. Verify the decoders

Three correctness checks that BLEU cannot give you.

In [ ]:
!python verify_decoding.py --preset base  --checkpoint-dir "{CKPT_DIR}"
!python verify_decoding.py --preset small --checkpoint-dir "{CKPT_DIR}"

## 6. Learning curves

The epoch where validation loss turns upward while training loss keeps falling
is the overfitting point. Name that epoch explicitly in the report.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, preset in zip(axes, ['base', 'small']):
    p = Path(CKPT_DIR) / preset / 'history.json'
    if not p.exists():
        ax.set_title(f'{preset}: no history found')
        continue
    h = json.loads(p.read_text())
    ep = [r['epoch'] for r in h]
    ax.plot(ep, [r['train_loss'] for r in h], 'o-', label='train')
    ax.plot(ep, [r['val_loss'] for r in h], 's-', label='validation')
    best = min(h, key=lambda r: r['val_loss'])
    ax.axvline(best['epoch'], ls='--', c='grey', lw=1)
    ax.annotate(f"best epoch {best['epoch']}\nval {best['val_loss']:.3f}",
                (best['epoch'], best['val_loss']),
                textcoords='offset points', xytext=(10, 20), fontsize=9)
    ax.set_title(f"{preset}  ({len(h)} epochs)")
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('artifacts/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Results table for the report

In [ ]:
import json

with open('artifacts/milestone3_results.json') as f:
    r = json.load(f)

print(f"{'model':<8}{'params':>12}{'val loss':>10}{'epochs':>8}"
      f"{'greedy':>9}{'beam':>9}{'gain':>8}{'beam s/s':>10}")
print('-' * 74)
for k, v in r.items():
    g = v['decoding']['greedy']['metrics']
    b = v['decoding']['beam']['metrics']
    print(f"{k:<8}{v['parameters']:>12,}{v['best_val_loss']:>10.4f}"
          f"{v['epochs_run']:>8}{g['bleu']:>9}{b['bleu']:>9}"
          f"{b['bleu'] - g['bleu']:>+8.2f}{b['sents_per_sec']:>10}")

for k, v in r.items():
    print(f"\n--- {k}: sample output ---")
    for gh, bh in zip(v['decoding']['greedy']['samples'][:3],
                      v['decoding']['beam']['samples'][:3]):
        print(f"  greedy: {gh}")
        print(f"  beam  : {bh}\n")

## 8. Copy artifacts back to Drive

Colab's local disk is wiped when the session ends. `milestone3_results.json`
and `learning_curves.png` are what you write the report from.

In [ ]:
!cp -r artifacts "{DRIVE_ROOT}/"
!ls -la "{DRIVE_ROOT}/artifacts"